In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import copy

In [2]:
from pandas import read_csv

from sklearn.preprocessing import MinMaxScaler, RobustScaler, PowerTransformer, FunctionTransformer
from sklearn.model_selection import KFold, train_test_split, GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, make_scorer
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer


from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

# Definition of Functions

### Custom-made transformations for the pipeline

In [3]:
class eliminatingCorrelation(BaseEstimator, TransformerMixin):
    def __init__(self, x1_name, x2_name, final_name):
        self.x1_name = x1_name
        self.x2_name = x2_name
        self.final_name = final_name
        self.model = LinearRegression()

    def fit(self, X, y=None):
        x1 = X[[self.x1_name]]
        x2 = X[self.x2_name]
        self.model.fit(x1, x2)
        return self
    
    def transform(self, X):
        x1 = X[[self.x1_name]]
        x2 = X[self.x2_name]
        residuals = x2 - self.model.predict(x1)
        X_new = X.copy()
        X_new.drop(self.x2_name, axis=1, inplace=True)
        X_new[self.final_name] = residuals

        return X_new 

def divideTB_DB(X):
    X_new = X.copy()
    if type(X_new) == np.ndarray: X_new[2] = X_new[2] / X_new[1].replace(0, 1e-6)
    if type(X_new) == pd.DataFrame: X_new["DB"] = X_new["DB"] / X_new["TB"].replace(0, 1e-6)
    return X_new

### Functions for data obtaining

In [14]:
def generate_pred_data():
    pred_data = read_csv("liver-patient-classification/test_data_ILDS.csv", delimiter = ',', header=None)
    pred_data.columns = ["Age", "Female", "TB", "DB", "Alkphos", "Sgpt", "Sgot", "TP", "ALB", "A/R"]
    columns = ["Age", "TB", "DB", "Alkphos", "Sgpt", "Sgot", "TP", "ALB", "A/R", "Female"]
    pred_data = pred_data[columns]
    return pred_data

def create_output(model, file = 'output.csv'):
    pred_data = generate_pred_data()
    y_new = chosen_model.predict(pred_data)
    y_pred = [[i+1, val] for i, val in enumerate(y_new)]
    final_result = pd.DataFrame(y_pred)
    final_result.columns = ["ID", "Label"]
    final_result.to_csv(file, index=False)

def generateX(data, test_size = 0.2, random_state=42):
    X = data.copy()
    X["Female"] = X["Female"].astype(int)
    X.drop("Target", axis=1, inplace=True)
    y = data["Target"]
    
    cols = ["Age", "TB", "DB", "Alkphos", "Sgpt", "Sgot", "TP", "ALB", "A/R", "Female"]
    X = X[cols]
            
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size,    
        stratify=y,       
        random_state=random_state 
    )
    X_train.shape
    X_train.head()
    return X_train, X_val, y_train, y_val

def outlier_treatment(data):
    numerical_columns = ["Age", "TB", "DB", "Alkphos", "Sgpt", "Sgot", "TP", "ALB", "A/R"]
    OutlierScaler = RobustScaler()
    ScaledOutlierX = OutlierScaler.fit_transform(data[numerical_columns])
    
    local_outlier_factor = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
    result = local_outlier_factor.fit_predict(ScaledOutlierX)
    
    outliers = result==-1
    no_outliers = result == 1
    print(f'The model determined that there are {np.sum(outliers)} outliers')
    print(f'The number of positive cases removed is {data[outliers & (data["Target"]==1)].count()[0]}')
    
    NO_data = data[no_outliers]
    return NO_data

def generate_train_data():
    data = read_csv("liver-patient-classification/train_features_ILDS.csv", delimiter = ',', header=None)
    data.columns = ["Age", "Female", "TB", "DB", "Alkphos", "Sgpt", "Sgot", "TP", "ALB", "A/R"]
    
    target = read_csv("liver-patient-classification/train_labels_ILDS.csv", header=None)
    data["Target"] = target
    columns = ["Age", "TB", "DB", "Alkphos", "Sgpt", "Sgot", "TP", "ALB", "A/R", "Female", "Target"]
    data = data[columns]
    return data

### Function for pipeline score

In [23]:
def model_evaluation(model1, model_name, sampler1, sampler_name, preprocessing, prep_name, param_grid,
                     X_train, y_train, X_test, y_test, model_list):
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scorer = make_scorer(f1_score)

    pipeline_list = []
    
    #Creating the Pipeline. It consist of eliminating Correlation between TB and DB, and a yeo-Johnson
    if preprocessing is not None:
        #print(type(preprocessing))
        for item in preprocessing:
            #print(item)
            pipeline_list.append(item)
    
    if sampler1 is not None:
        sampler = clone(sampler1)
        pipeline_list.append(("sampler", sampler))

    model = clone(model1)
    pipeline_list.append(("clf", model))
    pipeline = ImbPipeline(pipeline_list)

    #Creating the parameter grid
    pipeline_param_grid = {f'clf__{k}': v for k, v in param_grid.items()}    

    #print(pipeline)
    
    # Grid search
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=pipeline_param_grid,
        cv=cv,
        scoring=scorer,
        n_jobs=-1,
        verbose=0
    )

    grid.fit(X_train, y_train)
    
    #Results
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    best_params = {
        k.replace('clf__', ''): v
        for k, v in grid.best_params_.items()
        if k.startswith('clf__')
    }

    list_results = [prep_name, sampler_name, model_name,
        best_params, accuracy_score(y_val, y_pred),
        precision_score(y_val, y_pred),
        recall_score(y_val, y_pred),
        f1_score(y_val, y_pred, pos_label=0),
        f1_score(y_val, y_pred, pos_label=1),
        f1_score(y_val, y_pred), best_model
        ]
    model_list.append(list_results)


#Creates Preprocessing pipelines as a list
def create_prep_pipelines(d_Corr, d_Pwr, d_Scaler):
    pipeline_list = {}
    #keep_df = FunctionTransformer(lambda X: X if isinstance(X, pd.DataFrame) else pd.DataFrame(X))
    for corr, n_corr in d_Corr.items():
        for pwr, n_pwr in d_Pwr.items():
            for scaler, n_scaler in d_Scaler.items():
                pipe = [
                    #("keep_df", clone(keep_df)),
                    ("corr", clone(corr)),
                    ("pwr", clone(pwr)),
                    ("scaler", clone(scaler))
                ]
                name = f'{n_corr}, {n_pwr}, {n_scaler}'
                pipeline_list[name] = pipe
    return pipeline_list

def try_modelSampling(model, model_name, sampler, sampler_name, pipeline_list, param_grid,
                      X_train, y_train, X_test, y_test):
    columns = ["Preprocessing", "Sampler", "Model", "Parameters", "Accuracy", 
               "Precision", "Recall", "F1_class_0", "F1_class_1", "F1", "model"]
    
    results = []
    #results
    for message, prep_pipe in pipeline_list.items():
        #print(prep_pipe)
        model_evaluation(model, model_name, sampler, sampler_name, prep_pipe, message, param_grid,
                         X_train, y_train, X_test, y_test, results)
    results = pd.DataFrame(results, columns=columns)
    results = results.sort_values(by="F1", ascending=False)

    return results

# Model Definitions

### Preprocessing models

In [18]:
# Gaussian Transformations
yeojohnson_transform = ColumnTransformer(
    transformers=[
        ('yeojohnson', PowerTransformer(method='yeo-johnson'), [i for i in range(8)])
    ],
    remainder='passthrough'  # Leave non-numeric columns unchanged
)

identity = FunctionTransformer(lambda X: X)

d_Pwr_total = {yeojohnson_transform: "Yeo-Johnson",
        identity: "Identity"
        }

# Scalers
scalerMinMax = ColumnTransformer(
    transformers=[
        ('MinMax', MinMaxScaler(), [i for i in range(8)])
    ],
    remainder='passthrough'  # Leave non-numeric columns unchanged
)

scalerRobust = ColumnTransformer(
    transformers=[
        ('MinMax', RobustScaler(), [i for i in range(8)])
    ],
    remainder='passthrough'  # Leave non-numeric columns unchanged
)

d_Scaler_total = {scalerMinMax: "MinMax",
           scalerRobust: "Robust"
           }


# Eliminating correlation
linearCorr = eliminatingCorrelation("TB", "DB", "DB")
identity = FunctionTransformer(lambda X: X)
division = FunctionTransformer(lambda X: divideTB_DB(X))


d_Corr_total = {linearCorr: "Linear Regression",
         identity: "Identity",
          division: "Division"
         }

### Sampling Models

In [7]:
smotenc = SMOTENC(categorical_features=[9])

# Training different models

We'll choose different combinations of sampling and classification models. Our program runs through all the desired types of preprocessing and chooses the best hyper-parameters

In [17]:
data = generate_train_data()
NO_data = outlier_treatment(data)
X_train, X_val, y_train, y_val = generateX(NO_data)

X_train.head()

The model determined that there are 24 outliers
The number of positive cases removed is 2


/tmp/ipykernel_6169/4121271689.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f'The number of positive cases removed is {data[outliers & (data["Target"]==1)].count()[0]}')


,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/R,Female
219,26,7.1,3.3,258,80,113,6.2,2.9,0.8,0
145,45,2.3,1.3,282,132,368,7.3,4.0,1.2,0
13,73,1.8,0.9,220,20,43,6.5,3.0,0.8,0
89,51,2.9,1.2,189,80,125,6.2,3.1,1.0,0
358,42,16.4,8.9,245,56,87,5.4,2.0,0.5,0


## Logistic Regression + SMOTENC

In [19]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg_grid = {
    'C': np.logspace(-3, 3, 13),
    'penalty':['l2'],
    'solver': ['liblinear', 'lbfgs']
}

In [25]:
list_prep_pipeline = create_prep_pipelines(d_Corr_total, d_Pwr_total, d_Scaler_total)
result = try_modelSampling(logreg, "LogisticRegression", smotenc, "SMOTENC", list_prep_pipeline, logreg_grid,
                          X_train, y_train, X_val, y_val)
result.head()

,Preprocessing,Sampler,Model,Parameters,Accuracy,Precision,Recall,F1_class_0,F1_class_1,F1,model
0,"Linear Regression, Yeo-Johnson, MinMax",SMOTENC,LogisticRegression,"{'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}",0.670455,0.463415,0.730769,0.733945,0.567164,0.567164,"(eliminatingCorrelation(final_name='DB', x1_na..."
5,"Identity, Yeo-Johnson, Robust",SMOTENC,LogisticRegression,"{'C': 0.01, 'penalty': 'l2', 'solver': 'liblin...",0.670455,0.463415,0.730769,0.733945,0.567164,0.567164,(FunctionTransformer(func=<function <lambda> a...
9,"Division, Yeo-Johnson, Robust",SMOTENC,LogisticRegression,"{'C': 0.01, 'penalty': 'l2', 'solver': 'liblin...",0.670455,0.463415,0.730769,0.733945,0.567164,0.567164,(FunctionTransformer(func=<function <lambda> a...
3,"Linear Regression, Identity, Robust",SMOTENC,LogisticRegression,"{'C': 0.03162277660168379, 'penalty': 'l2', 's...",0.625000,0.428571,0.807692,0.673267,0.560000,0.560000,"(eliminatingCorrelation(final_name='DB', x1_na..."
8,"Division, Yeo-Johnson, MinMax",SMOTENC,LogisticRegression,"{'C': 0.0031622776601683794, 'penalty': 'l2', ...",0.659091,0.452381,0.730769,0.722222,0.558824,0.558824,(FunctionTransformer(func=<function <lambda> a...
